<a href="https://colab.research.google.com/github/jamesreddeviltna/Electric-Field-Simulation/blob/main/Fields.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Electric Field Simulation

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline

def calculate_e_field(x, y, sources):
    """
    Calculate the 2D electric field components Ex and Ey.
    Each source is a dict: {'x': val, 'y': val, 'q': val, 'r0': val}
    where 'r0' is the finite size/softening radius.
    """
    Ex = np.zeros_like(x, dtype=float)
    Ey = np.zeros_like(y, dtype=float)

    for src in sources:
        dx = x - src['x']
        dy = y - src['y']
        r2 = dx**2 + dy**2

        # Softening factor to avoid singularity at r -> 0
        # E ~ q * r / (r^2 + r0^2)^(1.5)
        r0_sq = src['r0']**2
        denom = (r2 + r0_sq)**1.5

        Ex += src['q'] * dx / denom
        Ey += src['q'] * dy / denom

    return Ex, Ey

# Setup the widgets
num_sources_slider = widgets.IntSlider(value=2, min=1, max=5, step=1, description='Num Sources')
ui_container = widgets.VBox([])
plot_output = widgets.Output()

def update_app(*args):
    num_sources = num_sources_slider.value
    source_widgets = []

    # Generate configuration widgets dynamically for each source
    for i in range(num_sources):
        # Standard starting values to make it look interesting initially
        q_init = 1.0 if i % 2 == 0 else -1.0
        x_init = -2.0 + i * 1.5
        y_init = 0.0

        header = widgets.HTML(f"<b>Source {i+1}</b>")
        x_w = widgets.FloatSlider(value=x_init, min=-5.0, max=5.0, step=0.1, description='X Pos')
        y_w = widgets.FloatSlider(value=y_init, min=-5.0, max=5.0, step=0.1, description='Y Pos')
        q_w = widgets.FloatSlider(value=q_init, min=-5.0, max=5.0, step=0.1, description='Charge')
        r0_w = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='Size (r0)')

        src_box = widgets.VBox([header, x_w, y_w, q_w, r0_w], layout=widgets.Layout(border='1px solid #ccc', padding='5px', margin='5px'))
        source_widgets.append(src_box)

    controls_box = widgets.GridBox(source_widgets, layout=widgets.Layout(grid_template_columns='repeat(auto-fit, minmax(200px, 1fr))'))

    # Function to trigger plot update
    def trigger_plot(*args_plot):
        with plot_output:
            plot_output.clear_output(wait=True)

            # Extract values from controls
            sources = []
            for box in source_widgets:
                children = box.children
                sources.append({
                    'x': children[1].value,
                    'y': children[2].value,
                    'q': children[3].value,
                    'r0': children[4].value
                })

            # Build coordinate grid
            nx, ny = 40, 40
            x = np.linspace(-6, 6, nx)
            y = np.linspace(-6, 6, ny)
            X, Y = np.meshgrid(x, y)

            Ex, Ey = calculate_e_field(X, Y, sources)

            fig, ax = plt.subplots(figsize=(8, 8))

            # Plot vector field using streamplot for visual flow
            color_intensity = np.log(np.sqrt(Ex**2 + Ey**2) + 1e-3)
            ax.streamplot(X, Y, Ex, Ey, color=color_intensity, cmap='viridis', linewidth=1.5, density=1.2, arrowstyle='->')

            # Draw the charge sources
            for src in sources:
                color = 'red' if src['q'] > 0 else 'blue' if src['q'] < 0 else 'gray'
                size_factor = abs(src['q']) * 100 + 50
                # Solid circle to show position and finite size
                circle = plt.Circle((src['x'], src['y']), src['r0'], color=color, alpha=0.3)
                ax.add_patch(circle)
                ax.scatter(src['x'], src['y'], color=color, s=size_factor, edgecolor='black', zorder=5)
                ax.text(src['x'] + 0.15, src['y'] + 0.15, f"{src['q']:+.1f}", fontsize=10, weight='bold', color='black', bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))

            ax.set_xlim(-6, 6)
            ax.set_ylim(-6, 6)
            ax.set_aspect('equal')
            ax.set_title('2D Electric Field Map (Finite-Size Sources)', fontsize=14)
            ax.set_xlabel('X coordinate')
            ax.set_ylabel('Y coordinate')
            ax.grid(True, linestyle='--', alpha=0.5)
            plt.show()

    # Bind triggers to redraw plot on any parameter change
    for box in source_widgets:
        for child in box.children[1:]:
            child.observe(trigger_plot, names='value')

    ui_container.children = [controls_box]
    trigger_plot()

num_sources_slider.observe(update_app, names='value')
update_app()

display(widgets.VBox([num_sources_slider, ui_container, plot_output]))